# 01 — Dataset Preparation
**AI Travel Assistant Chatbot — SRH Applied AI Project**

This notebook:
1. Downloads the **Bitext Travel LLM Chatbot Training Dataset** from HuggingFace Hub
2. Explores and cleans the data
3. Converts it to instruction-tuning (SFT) format
4. Splits into train / test
5. Saves as `travel_sft_train.jsonl` and `travel_sft_test.jsonl`

Dataset: [`bitext/Bitext-travel-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-travel-llm-chatbot-training-dataset)

> Run this notebook on Google Colab (free CPU is enough — no GPU needed for data prep).

## Step 1 — Install dependencies

In [ ]:
!pip install -q datasets pandas huggingface_hub

## Step 2 — Load the dataset from HuggingFace

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Loading Bitext travel chatbot dataset from HuggingFace...")

# This is a public dataset — no token required
raw_ds = load_dataset("bitext/Bitext-travel-llm-chatbot-training-dataset", split="train")

print(f"Total rows loaded: {len(raw_ds)}")
print(f"Columns: {raw_ds.column_names}")
print("\nFirst example:")
print(raw_ds[0])

## Step 3 — Explore the data

In [ ]:
df = raw_ds.to_pandas()

print("=== Dataset Shape ===")
print(df.shape)

print("\n=== Column Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Sample rows ===")
df.head(5)

In [ ]:
# Check intent distribution (the Bitext dataset uses 'intent' column)
if 'intent' in df.columns:
    print("Intent distribution (top 20):")
    print(df['intent'].value_counts().head(20))
    print(f"\nTotal unique intents: {df['intent'].nunique()}")
elif 'category' in df.columns:
    print("Category distribution:")
    print(df['category'].value_counts())
else:
    print("Columns found:", df.columns.tolist())
    print(df.head(3).to_string())

## Step 4 — Clean the data

In [ ]:
import re

# The Bitext dataset has columns: instruction, response (and optionally intent, category, tags)
# We map to the standard field names used in our pipeline

# Detect column names
instruction_col = None
response_col = None

for col in ['instruction', 'question', 'input', 'prompt', 'utterance']:
    if col in df.columns:
        instruction_col = col
        break

for col in ['response', 'answer', 'output', 'reply']:
    if col in df.columns:
        response_col = col
        break

print(f"Detected: instruction='{instruction_col}', response='{response_col}'")

# Keep only needed columns
df_clean = df[[instruction_col, response_col]].copy()
df_clean.columns = ['instruction', 'response']

print(f"Before cleaning: {len(df_clean)} rows")

# 1. Drop rows with missing instruction or response
df_clean = df_clean.dropna(subset=['instruction', 'response'])

# 2. Drop rows where instruction or response is empty string
df_clean = df_clean[df_clean['instruction'].str.strip().str.len() > 5]
df_clean = df_clean[df_clean['response'].str.strip().str.len() > 10]

# 3. Strip whitespace
df_clean['instruction'] = df_clean['instruction'].str.strip()
df_clean['response'] = df_clean['response'].str.strip()

# 4. Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['instruction'])

# 5. Remove very short responses (likely boilerplate)
df_clean = df_clean[df_clean['response'].str.len() >= 20]

# 6. Remove very long responses (can cause OOM during training)
df_clean = df_clean[df_clean['response'].str.len() <= 2000]

df_clean = df_clean.reset_index(drop=True)
print(f"After cleaning: {len(df_clean)} rows")
print("\nSample cleaned instruction:")
print(df_clean['instruction'].iloc[0])
print("\nSample cleaned response:")
print(df_clean['response'].iloc[0])

## Step 5 — Convert to instruction-tuning (SFT) format

We use the **chat messages format** that Gemma's `apply_chat_template` understands:
```json
{
  "messages": [
    {"role": "user", "content": "<travel question>"},
    {"role": "assistant", "content": "<ideal answer>"}
  ]
}
```

In [ ]:
def to_sft_record(instruction: str, response: str) -> dict:
    """Wrap a Q/A pair into chat-messages SFT format."""
    return {
        "messages": [
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": response}
        ]
    }

records = [
    to_sft_record(row['instruction'], row['response'])
    for _, row in df_clean.iterrows()
]

print(f"Total SFT records: {len(records)}")
print("\nExample record:")
import json
print(json.dumps(records[0], indent=2))

## Step 6 — Train / Test split

In [ ]:
import random

random.seed(42)
random.shuffle(records)

# 90% train, 10% test — keep test set at max 200 examples for fast evaluation
n_test = min(200, int(len(records) * 0.10))
n_train = len(records) - n_test

train_records = records[:n_train]
test_records  = records[n_train:]

print(f"Train examples: {len(train_records)}")
print(f"Test examples:  {len(test_records)}")

## Step 7 — Save as JSONL files

In [ ]:
import pathlib, json

OUT_DIR = pathlib.Path("data")
OUT_DIR.mkdir(exist_ok=True)

def save_jsonl(records: list, path: pathlib.Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records → {path}")

save_jsonl(train_records, OUT_DIR / "travel_sft_train.jsonl")
save_jsonl(test_records,  OUT_DIR / "travel_sft_test.jsonl")

# Also save the full clean dataframe as CSV for inspection
df_clean.to_csv(OUT_DIR / "travel_dataset_clean.csv", index=False)
print("Saved travel_dataset_clean.csv")

## Step 8 — Dataset statistics summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

instr_lengths = df_clean['instruction'].str.len()
resp_lengths  = df_clean['response'].str.len()

print("=" * 45)
print("       DATASET STATISTICS SUMMARY")
print("=" * 45)
print(f"Total examples        : {len(df_clean):>8}")
print(f"Train examples        : {len(train_records):>8}")
print(f"Test examples         : {len(test_records):>8}")
print("-" * 45)
print(f"Avg instruction length: {instr_lengths.mean():>8.0f} chars")
print(f"Avg response length   : {resp_lengths.mean():>8.0f} chars")
print(f"Min response length   : {resp_lengths.min():>8} chars")
print(f"Max response length   : {resp_lengths.max():>8} chars")
print("=" * 45)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(instr_lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Instruction Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

axes[1].hist(resp_lengths, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Response Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(OUT_DIR / "dataset_length_distribution.png", dpi=120)
plt.show()
print("Plot saved to data/dataset_length_distribution.png")

## Done!

Files produced:
- `data/travel_sft_train.jsonl` — training set for fine-tuning
- `data/travel_sft_test.jsonl` — test set for evaluation
- `data/travel_dataset_clean.csv` — human-readable version
- `data/dataset_length_distribution.png` — length plots

**Next step:** Open `02_finetune_gemma.ipynb` to fine-tune Gemma 2B on this dataset.